In [7]:

from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#3 创建PostgreSQL数据库连接
DB_URL = os.getenv("DB_URL")
with PostgresSaver.from_conn_string(DB_URL) as myCheckpointer:
    #初始化数据库
    myCheckpointer.setup()
    #4初始化1个agent实例
    myAgent=create_agent(
        model=myModel,
        name="琳琳",
        #关键设置，让Agent具备了存储的能力
        checkpointer=myCheckpointer,
    )
    #4 创建1个config文件，保证同1个thread id是共享记忆的
    myConfig={
        "configurable":{
            "thread_id":"Aureliano"
        }
    }
    response=myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [ SystemMessage("你是1个专业的Ai助手"),
                  HumanMessage("你好，我是Aureliano"),
                ],
        },
        config=myConfig
    )

    for msg in response["messages"]:
        msg.pretty_print()



    response2=myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                  HumanMessage("你知道我是谁吗？"),
                ],
        },
        config=myConfig
    )

    for msg in response2["messages"]:
        msg.pretty_print()



================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

你好，我是Aureliano
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你。有什么我可以帮你的吗？
================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

你好，我是Aureliano
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你。有什么我可以帮你的吗？
================================ Human Message =================================

你知道我是谁吗？
================================== Ai Message ==================================
Name: 琳琳

根据你刚才告诉我的，你是 Aureliano。

不过，作为一个人工智能，我并不知道你的真实身份、背景或更多的个人信息，我只知道你在这个对话中愿意分享给我的内容。

有什么我可以帮你的吗，Aureliano？


In [10]:
from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#3 创建PostgreSQL数据库连接
DB_URL = os.getenv("DB_URL")
with PostgresSaver.from_conn_string(DB_URL) as myCheckpointer:
    #初始化数据库
    myCheckpointer.setup()
    #4初始化1个agent实例
    myAgent=create_agent(
        model=myModel,
        name="琳琳",
        #关键设置，让Agent具备了存储的能力
        checkpointer=myCheckpointer,
    )
    #4 创建1个config文件，保证同1个thread id是共享记忆的
    myConfig={
        "configurable":{
            "thread_id":"Aureliano"
        }
    }
    response=myAgent.invoke(
            {
                # 严格校验，使用message少了1个s就会报错
                "messages":
                    [
                      HumanMessage("距离上次找你已经很久了，你现在还知道我是谁吗"),
                    ],
            },
            config=myConfig
        )

    for msg in response["messages"]:
            msg.pretty_print()

================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

你好，我是Aureliano
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你。有什么我可以帮你的吗？
================================ Human Message =================================

你知道我是谁吗？
================================== Ai Message ==================================
Name: 琳琳

根据你刚才告诉我的，你是 Aureliano。

不过，作为一个人工智能，我并不知道你的真实身份、背景或更多的个人信息，我只知道你在这个对话中愿意分享给我的内容。

有什么我可以帮你的吗，Aureliano？
================================ Human Message =================================

距离上次找你已经很久了，你现在还知道我是谁吗
================================== Ai Message ==================================
Name: 琳琳

如果你指的是我们**当前这段对话**里，我记得你刚才告诉我：你叫 **Aureliano**。

但如果你指的是**很久之前另一次对话**，那我通常无法保留或访问之前的聊天记录，所以不能确定你当时告诉过我什么。

如果你愿意再介绍一下自己，我可以接着记住你在本次对话里告诉我的内容。


总而言之 InMemerySaver()会将状态持久化到内存，进程结束或者重建Saver(),则历史状态丢失。 但是基于外部存储介质例如PostgreSQL的持久化器，其存储的状态不会随进程的终止而丢失，只要**不显式的删除**历史状态，即可通过thread_id 加载历史状态